In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as sk

In [20]:
pay = pd.read_csv('../data/olist_order_payments_dataset.csv')
# pay.info()
# pay.isnull().sum()
df_order_payments = pay.copy()

review = pd.read_csv('../data/olist_order_reviews_dataset.csv')
# review.info()
# review.isnull().sum()
df_order_reviews = review.copy()

In [21]:
# print(df_order_payments.duplicated().sum()) #0

In [22]:
order = pd.read_csv("../data/olist_orders_dataset.csv")
order_copy = order.copy()
df_orders = order_copy

# order_copy.info()
# order_copy.isnull().sum()


#형변환

##결측X / (시각데이터 날림 코드 밑에 주석처리)
#date type 변환 (str -> date) / 연산 필요시 date 재변환 가능성 있음

#제거 전 확인
#time_part = order_copy['order_estimated_delivery_date'].str[-8:] 
#no_time = (time_part == '00:00:00').sum()
#print(no_time) #99441 총 행 갯수와 동일 모두 00:00:00 _제거 진행

#시분초 제거 및 타입 전환 코드 
# order_copy['order_estimated_delivery_date'] = pd.to_datetime(
#     order_copy['order_estimated_delivery_date'], 
#     format='%Y-%m-%d %H:%M:%S'
# ).dt.date
#order_copy['order_estimated_delivery_date'].head(20)

''' 시각 데이터 모든행 '00:00:00' '''
df_orders['order_estimated_delivery_date'] = pd.to_datetime(
    df_orders['order_estimated_delivery_date'], 
    format='%Y-%m-%d %H:%M:%S'
)

#형변환
##결측X 
#date type 변환 (str -> date) 
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])


#형변환 
##결측 O 
#date type 변환 (str -> date) 
cols_na = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

df_orders[cols_na] = df_orders[cols_na].apply(
    pd.to_datetime,
    errors="coerce"
)

#df_orders.info()


In [23]:
#데이터 정제- 불필요 행 제거

#order_status 칼럼 - 결측X, 분석 목표인 배달 요소와 관련 없는 status 분리하여 분석 진행이 용이할 것으로 판단
#'배송 품질' 일단 Olist와 직결된 물류 배송에 국한하여 정의/ 상품 준비단계에서 소요되는 날짜 논외
core_statuses = ['delivered', 'shipped', 'canceled']
df_orders = df_orders[df_orders['order_status'].isin(core_statuses)]
df_orders['order_status'].value_counts()

df_orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 155
order_delivered_carrier_date      552
order_delivered_customer_date    1734
order_estimated_delivery_date       0
dtype: int64

In [24]:
#결측 처리

# 1. 이상치 분리: 
# order_status 'canceled'인데 order_delivered_customer_date 값 있음 #6개 
# >> review message 보니까 제거하면 안되는 case, 새로운 데이터프레임으로 분리 저장
con_0 = (
    (df_orders["order_status"] == "canceled") &
    (df_orders["order_delivered_customer_date"].notna())
)
df_canceled_with_delivery = df_orders[con_0].copy()

# 2. df_orders에서는 해당 행들 삭제
df_orders = df_orders[~con_0]

# 3. 결과 확인
print(f"분리된 행: {df_canceled_with_delivery.shape[0]}개")
print(f"df_orders 남은 행: {df_orders.shape[0]}개")
print(f"\ndf_canceled_with_delivery:")
print(df_canceled_with_delivery)

df_orders.isnull().sum()

분리된 행: 6개
df_orders 남은 행: 98204개

df_canceled_with_delivery:
                               order_id                       customer_id  \
2921   1950d777989f6a877539f53795b4c3c3  1bccb206de9f0f25adc6871a1bcf77b2   
8791   dabf2b0e35b423f94618bf965fcb7514  5cdec0bb8cbdf53ffc8fdc212cd247c6   
58266  770d331c84e5b214bd9dc70a10b829d0  6c57e6119369185e575b36712766b0ef   
59332  8beb59392e21af5eb9547ae1a9938d06  bf609b5741f71697f65ce3852c5d2623   
92636  65d1e226dfaeb8cdc42f665422522d14  70fc57eeae292675927697fe03ad3ff5   
94399  2c45c33d2f9cb8ff8b1c86cc28c11c30  de4caa97afa80c8eeac2ff4c8da5b72e   

      order_status order_purchase_timestamp   order_approved_at  \
2921      canceled      2018-02-19 19:48:52 2018-02-19 20:56:05   
8791      canceled      2016-10-09 00:56:52 2016-10-09 13:36:58   
58266     canceled      2016-10-07 14:52:30 2016-10-07 15:07:10   
59332     canceled      2016-10-08 20:17:50 2016-10-09 14:34:30   
92636     canceled      2016-10-03 21:01:41 2016-10-04 10:18:57 

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 155
order_delivered_carrier_date      552
order_delivered_customer_date    1734
order_estimated_delivery_date       0
dtype: int64

In [25]:
# order_status 'canceled'인데 order_delivered_customer_date 값 있는 6개 데이터, 분리한 df
df_canceled_with_delivery

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,1bccb206de9f0f25adc6871a1bcf77b2,canceled,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09
8791,dabf2b0e35b423f94618bf965fcb7514,5cdec0bb8cbdf53ffc8fdc212cd247c6,canceled,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30
58266,770d331c84e5b214bd9dc70a10b829d0,6c57e6119369185e575b36712766b0ef,canceled,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29
59332,8beb59392e21af5eb9547ae1a9938d06,bf609b5741f71697f65ce3852c5d2623,canceled,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30
92636,65d1e226dfaeb8cdc42f665422522d14,70fc57eeae292675927697fe03ad3ff5,canceled,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,de4caa97afa80c8eeac2ff4c8da5b72e,canceled,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08


In [26]:
# 결측 처리
# order_status 'delivered', 'order_approved_at'컬럼 결측 대체
# 조건 정의
con_1 = (
    (df_orders['order_status'] == 'delivered') &
    (df_orders['order_approved_at'].isna())
)

# 결측 order_purchase_timestamp로 대체
df_orders.loc[con_1, 'order_approved_at'] = df_orders.loc[con_1, 'order_purchase_timestamp']

# 결과 확인
print(con_1.sum()) #행갯수
print(df_orders.loc[con_1, ['order_approved_at', 'order_purchase_timestamp']].head())

14
        order_approved_at order_purchase_timestamp
5323  2017-02-18 14:40:00      2017-02-18 14:40:00
16567 2017-02-18 12:45:31      2017-02-18 12:45:31
19031 2017-02-18 13:29:47      2017-02-18 13:29:47
22663 2017-02-18 16:48:35      2017-02-18 16:48:35
23156 2017-02-17 13:05:55      2017-02-17 13:05:55


In [27]:
# order_status 'delivered', 'order_delivered_carrier_date'컬럼 결측 대체
# 조건 정의
con_2 = (
    (df_orders['order_status'] == 'delivered') &
    (df_orders['order_delivered_carrier_date'].isna())
)

# 해당 행들만 order_approved_at로 대체
df_orders.loc[con_2, 'order_delivered_carrier_date'] = df_orders.loc[con_2, 'order_approved_at']

# 결과 확인
print(con_2.sum()) #행갯수
print(df_orders.loc[con_2, ['order_delivered_carrier_date', 'order_approved_at']].head())

2
      order_delivered_carrier_date   order_approved_at
73222          2017-09-29 09:07:16 2017-09-29 09:07:16
92643          2017-05-25 23:30:16 2017-05-25 23:30:16


In [ ]:
# rows_before = len(df_orders)
# print(f"전 전체 행 수: {rows_before}")

# # 조건 정의: 'delivered' 상태인데 'order_delivered_customer_date' 결측
# con_3 = (
#     (df_orders['order_status'] == 'delivered') &
#     (df_orders['order_delivered_customer_date'].isna())
# )

# # 조건 해당 행 제거
# df_orders = df_orders[~con_3].copy()

# # 실행 후 전체 행 수
# rows_after = len(df_orders)
# print(f"코드 실행 후 전체 행 수: {rows_after}")

# # 줄어든 행 수 확인
# print(f"제거된 행 수: {rows_before - rows_after}")

코드 실행 전 전체 행 수: 98204
코드 실행 후 전체 행 수: 98196
제거된 행 수: 8


In [ ]:
# (1)-3. 
# order_status 'delivered', 'order_delivered_customer_date'컬럼 결측 제거

con_3 = (

    (df_orders['order_status'] == 'delivered') &
    (df_orders['order_delivered_customer_date'].isna())
)

# 조건 해당 행 제거
df_orders = df_orders[~con_3].copy()

In [29]:
# # order_status 'delivered', 'order_delivered_customer_date'컬럼 결측 reviews table 리뷰 작성 날짜로 대체
# # df_orders와 df_order_reviews(order_id 기준) 임의 merge
# df_orders = df_orders.merge(
#     df_order_reviews[['order_id', 'review_creation_date']],
#     on='order_id',
#     how='left'
# )

# # review_creation_date datetime 임의 변환
# df_orders['review_creation_date'] = pd.to_datetime(df_orders['review_creation_date'], errors='coerce')

# # 조건 정의
# con_3 = (
#     (df_orders['order_status'] == 'delivered') &
#     (df_orders['order_delivered_customer_date'].isna())
# )

# # 결측 채우기
# df_orders.loc[con_3, 'order_delivered_customer_date'] = df_orders.loc[con_3, 'review_creation_date']

# # 결과 확인
# print(con_3.sum()) #행갯수
# print(df_orders.loc[con_3, ['order_delivered_customer_date', 'review_creation_date']].head())


In [30]:
# # merge 후 불필요 컬럼 제거
# df_orders = df_orders.drop(columns=['review_creation_date'])

# # 제거 후 확인
# print(df_orders.loc[con_3, ['order_delivered_customer_date']].head())

In [31]:
df_orders.info()
df_orders.isnull().sum()

<class 'pandas.DataFrame'>
Index: 98204 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       98204 non-null  str           
 1   customer_id                    98204 non-null  str           
 2   order_status                   98204 non-null  str           
 3   order_purchase_timestamp       98204 non-null  datetime64[us]
 4   order_approved_at              98063 non-null  datetime64[us]
 5   order_delivered_carrier_date   97654 non-null  datetime64[us]
 6   order_delivered_customer_date  96470 non-null  datetime64[us]
 7   order_estimated_delivery_date  98204 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.7 MB


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 141
order_delivered_carrier_date      550
order_delivered_customer_date    1734
order_estimated_delivery_date       0
dtype: int64

In [32]:
# 조건 정의: shipped 상태 & order_delivered_customer_date 결측 정상_ "2099-01-01 00:00:00" 형태로 대체
con_4= (
    (df_orders['order_status'] == 'shipped') &
    (df_orders['order_delivered_customer_date'].isna())
)

# 정상 결측 값 정의
normal_blank = pd.Timestamp("2099-01-01 00:00:00")

# 결측 대체
df_orders.loc[con_4, 'order_delivered_customer_date'] = normal_blank

# 결과 확인
print("채운 행 개수:", con_4.sum())
print(df_orders.loc[con_4, ['order_id', 'order_delivered_customer_date']].head())

채운 행 개수: 1107
                             order_id order_delivered_customer_date
44   ee64d42b8cf066f35eac1cf57de1aa85                    2099-01-01
154  6942b8da583c2f9957e990d028607019                    2099-01-01
162  36530871a5e80138db53bcfd8a104d90                    2099-01-01
231  4d630f57194f5aba1a3d12ce23e71cd9                    2099-01-01
299  3b4ad687e7e5190db827e1ae5a8989dd                    2099-01-01


In [33]:
# 조건 정의: canceled 상태 & order_delivered_customer_date 결측 정상_ "2099-01-01 00:00:00" 형태로 대체
# 조건: canceled 상태
con_5 = df_orders['order_status'] == 'canceled'

# 결측 처리할 컬럼 리스트
cols_fill= [
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date'
]

# 한 번에 결측 처리
for col in cols_fill:
    df_orders.loc[con_5 & df_orders[col].isna(), col] = normal_blank

# 결과 확인
display(df_orders.loc[con_5, ['order_id', 'order_status'] + cols_fill].head())

,order_id,order_status,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
397,1b9ecfe83cdc259250e1a8aca174f0ad,canceled,2018-08-07 04:10:26,2099-01-01 00:00:00,2099-01-01
613,714fb133a6730ab81fa1d3c1b2007291,canceled,2018-01-26 21:58:39,2018-01-29 22:33:25,2099-01-01
1058,3a129877493c8189c59c60eb71d97c29,canceled,2018-01-25 13:50:20,2018-01-26 21:42:18,2099-01-01
1130,00b1cb0320190ca0daa2c88b35206009,canceled,2099-01-01 00:00:00,2099-01-01 00:00:00,2099-01-01
1801,ed3efbd3a87bea76c2812c66a0b32219,canceled,2099-01-01 00:00:00,2099-01-01 00:00:00,2099-01-01


In [34]:
df_orders.info()
df_orders.isnull().sum()

<class 'pandas.DataFrame'>
Index: 98204 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       98204 non-null  str           
 1   customer_id                    98204 non-null  str           
 2   order_status                   98204 non-null  str           
 3   order_purchase_timestamp       98204 non-null  datetime64[us]
 4   order_approved_at              98204 non-null  datetime64[us]
 5   order_delivered_carrier_date   98204 non-null  datetime64[us]
 6   order_delivered_customer_date  98196 non-null  datetime64[us]
 7   order_estimated_delivery_date  98204 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.7 MB


order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    8
order_estimated_delivery_date    0
dtype: int64

In [35]:
print(df_order_payments.duplicated().sum()) #0 #여기서는 중복행 없음...

0
